In [137]:
import csv
import pandas as pd
import json
import numpy as np
PATH = "C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/"
# PATH = "D:/Udemy/2026-Python3-ErikBanas/Section53_PythonForFinance/redo/"
tickers_to_skip = ['SEB']
missing_tickers = []

# pd.set_option('display.max_colwidth', 500)

In [138]:
# sec_df = pd.read_csv(PATH + 'sector-stocks/stock_sectors.csv')

# sectors = [
#     "Industrials"
# ]
indus_df = sec_df.loc[sec_df['Sector'] == "industrials"]
industrial = get_rois_for_stocks(indus_df, '2025-01-01','2025-12-31')
industrial = industrial.sort_values(by='Ticker')


0.3739642630035964 2025-01-02 2025-12-31
File for ticker RLGT doesn't exist
File for ticker CIX doesn't exist
File for ticker FJET doesn't exist
File for ticker EVI doesn't exist
File for ticker GENC doesn't exist
File for ticker ESP doesn't exist
File for ticker EGG doesn't exist
File for ticker FLYX doesn't exist


In [139]:
def get_history_indicator_values(ticker, data, indicator_id, num_items):
    roic = []
    try:
        roic = next((item for item in data if item["id"] == indicator_id), None) 
        if roic is None:
            return None
        roic = roic['value'] 
        roic = roic[:num_items]
        roic = roic[::-1]
    except Exception as e:
        print (f'Error in get_history_indicator_values.{ticker} {e}. Inidicator: {indicator_id}')
    else:
        return roic
        
    return None
    
def get_growth_rates(revenues):
    growth_rates = []

    for i in range(1, len(revenues)):
        previous = revenues[i - 1]
        current = revenues[i]
        growth = ((current - previous) / previous) * 100
        growth_rates.append(growth)
    
    return growth_rates
    
def get_df_from_csv(ticker):
    
    # Try to get the file and if it doesn't exist issue a warning
    try:

        if ticker not in tickers_to_skip:
            df = pd.read_csv(PATH + 'converted/' + ticker + '.csv')
            # df['date2'] = pd.DatetimeIndex(['date'])
            df = df.set_index(pd.to_datetime(df['date']))
            df.drop(columns=['date'], inplace=True)
            # df = df.loc[S_DATE_STR:E_DATE_STR]
            return df
        else:
            return None
        # df = delete_unnamed_cols(df)
        
    except FileNotFoundError:
        missing_tickers.append(ticker)
        print(f"File for ticker {ticker} doesn't exist")
        return None
        
def get_rois_for_stocks(stock_df, sdate, edate):

    tickers = []
    rois = []
    c = []
    for index, row in stock_df.iterrows():
        df = get_df_from_csv(row['ticker'])
        if df is None:
            pass
        else:
            mask = (df.index > sdate) & (df.index <= edate)
            
            if len(df.loc[mask]) > 0:
                
                sdate2, edate2 = get_valid_dates(df, sdate, edate)
                roi = roi_between_dates(df, sdate2, edate2)
                # if roi > 0:
                rois.append(roi)
                
                if row['ticker'] == 'AWI':
                    print (roi, sdate2, edate2)
                tickers.append(row['ticker'])
                c.append(row['company_name'])

    return pd.DataFrame({'Ticker' : tickers,  'Company': c , 'ROI' : rois})

def roi_between_dates(df, sdate, edate):
    try: 
        start_val = df.loc[sdate,'close'] 
        end_val = df.loc[edate,'close']
        # print(f"start_val: {start_val}")
        # print(f"end_val: {end_val}")
        
        roi = ((end_val - start_val) / start_val)
    except Exception:
        print("Data Corrupted")
    else:
        return roi

def get_valid_dates(df, sdate, edate):
    
    try:
        # mask = (df['date'] > sdate) & (df['date'] <= edate) 
        sm_df = df.loc[sdate:edate]
        if not sm_df.empty:
            sm_date = str(sm_df.index.min()).split(' ')[0]
            last_date = str(sm_df.index.max()).split(' ')[0]
            
            date_leading = str(sm_df.index.min()).split(' ')[0] #'-'.join(('0' if len(x) < 2 else '')+x for x in sm_date.split('-'))
            date_ending = str(sm_df.index.max()).split(' ')[0]  #'-'.join(('0' if len(x) < 2 else '')+x for x in last_date.split('-'))
            # print(date_leading, " ", date_ending)
        else:
            return None, None
    except Exception:
        print("Date Corrupted")
    else:
        return date_leading, date_ending

df = get_df_from_csv('AWI')
sdate, edate = get_valid_dates(df, '2025-01-01','2025-12-31')
roi_between_dates(df, sdate, edate)


np.float64(0.3739642630035964)

In [142]:
def get_formatted_data(sector_df):
    # data = {
    #     "AAPL": {
    #         "sector": "Technology",
    #         "pe": [25, 27, 24],
    #         "ev_ebitda": [18, 17, 16],
    #         "roic": [12, 13, 15],
    #         "revenue_growth": [5, 6, 7]
    #     },
    # }
    data = None    
    tickers, betas, rois, pes, roics, ev_ebitdas, growths, debt_ebitdas, oms = [], [], [], [], [], [], [], [], []
     
    for ticker in sector_df['Ticker'].values.tolist():
        
        try:
    
            with open(f"{PATH}dividends/{ticker}.json", 'r') as file:
                data = json.load(file)['data']

            if len(data) > 0:
                
                pe = get_history_indicator_values(ticker, data, 'price_earnings_fy_h', 3)
                roic = get_history_indicator_values(ticker, data, 'return_on_invested_capital_fy_h', 3)
                ev_ebitda = get_history_indicator_values(ticker, data, 'enterprise_value_ebitda_fy_h', 3)
                growth = get_history_indicator_values(ticker, data, 'total_revenue_fy_h', 4)
                
                debt = next((item for item in data if item["id"] == 'total_debt_h'), None)
                
                if debt is not None:      
                    debt = debt['value'] 
                    debt = debt[:3]
                    debt = debt[::-1]

                    ebitda = get_history_indicator_values(ticker, data, 'ebitda_fy_h', 3)
                    
                    debt_ebitda = pd.Series(debt) / pd.Series(ebitda)
                    debt_ebitda = debt_ebitda.tolist()

                    om = get_history_indicator_values(ticker, data, 'operating_margin_fy_h', 3)
                   
                    p = [n for n in pe if pd.isna(n)]
                    r = [n for n in roic if pd.isna(n)]
                    e = [n for n in ev_ebitda if pd.isna(n)]
                    o = [n for n in om if pd.isna(n)]
                    
                    if len(p) == 0 and len(r) == 0 and len(e) == 0 and len(o) == 0 and 0 not in growth:
                        rois.append(sector_df[sector_df['Ticker'] == ticker]['ROI'].item())
                        tickers.append(ticker)
                        pes.append(pe)
                        roics.append(roic)
                        ev_ebitdas.append(ev_ebitda)
                        growths.append(get_growth_rates(growth))
                        debt_ebitdas.append(debt_ebitda)
                        oms.append(om)
                    
        except Exception as e:
            print (f'123 Error ticker: {ticker} {e}')
    
    
    df_2 = pd.DataFrame({
        'ticker': tickers, 
        'roi': rois, 
        'pe': pes,
        'roic': roics,
        'ev_ebitda': ev_ebitdas,
        'revenue_growth': growths, 
        'debt_ebitda': debt_ebitdas,
        'om': oms,
        'sector': 'industrials'})
    
    
    df_2.set_index('ticker', inplace=True)
    dict_2 = df_2.to_dict(orient='index')
    df_2.to_json(PATH + 'industrials.json')

    return df_2, dict_2

In [143]:
df_3, dict_3 = get_formatted_data(industrial)
df_3[df_3.index == 'AWI']


123 Error ticker: BWLP Expecting ',' delimiter: line 5396 column 25 (char 166879)


,roi,pe,roic,ev_ebitda,revenue_growth,debt_ebitda,om,sector
ticker,,,,,,,,
AWI,0.373964,"[19.6903851160555, 23.4751843731314, 26.990383...","[18.5350898997431, 21.0029732408325, 22.950819...","[15.2830892801977, 17.8084507042254, 19.665820...","[5.036087908523234, 11.619827053736875, 12.111...","[1.5332097621254248, 1.3125166090884932, 1.291...","[18.1053119209389, 18.8905028705817, 19.755676...",industrials


In [144]:
def evaluate_stocks_by_sector(tickers, data, ticker_to_watch = ''):

    def trend_improving(values, higher_is_better=True):
        if not values or len(values) < 2:
            return False
        # if higher_is_better:
        #     return values[0] > values[1] and values[1] > values[2]
        # else:
        #     return values[0] < values[1] and values[1] < values[2]
        
        return values[-1] > values[0] if higher_is_better else values[-1] < values[0]

    # -----------------------------
    # STEP 1: Group by sector
    # -----------------------------
    sector_groups = {}

    for ticker in tickers:
        
        sector = data[ticker]["sector"]
        sector_groups.setdefault(sector, []).append(ticker)

    # -----------------------------
    # STEP 2: Compute sector averages
    # -----------------------------
    sector_averages = {}

    for sector, stocks in sector_groups.items():
        pe_vals, ev_vals, roic_vals, debt_ebitda_vals, oms  = [], [], [], [], []

        for t in stocks:
            d = data[t]
                
            if d.get("pe"):
                pe_vals.append(d["pe"][-1])
            if d.get("ev_ebitda"):
                ev_vals.append(d["ev_ebitda"][-1])                
            if d.get("roic"):
                roic_vals.append(d["roic"][-1])
            
            if d.get("debt_ebitda"):
                if not pd.isna(d["debt_ebitda"][-1]):
                    debt_ebitda_vals.append(d["debt_ebitda"][-1])

            if d.get("om"):
                if not pd.isna(d["om"][-1]):
                    oms.append(d["om"][-1])

            # if pd.isna(d["debt_ebitda"][-1]):
            #     print(t)

        sector_averages[sector] = {
            "pe": sum(pe_vals)/len(pe_vals) if pe_vals else None,
            "ev_ebitda": sum(ev_vals)/len(ev_vals) if ev_vals else None,
            "roic": sum(roic_vals)/len(roic_vals) if roic_vals else None,
            "debt_ebitda": sum(debt_ebitda_vals)/len(debt_ebitda_vals) if debt_ebitda_vals else None,
            "om": sum(oms)/len(oms) if oms else None,
        }

    # -----------------------------
    # STEP 3: Evaluate each stock
    # -----------------------------
    results = {}

    # for x in range(len(debt_ebitda_vals)):
    #     print (debt_ebitda_vals[x])
        
    # print (sum(debt_ebitda_vals), len(debt_ebitda_vals))
    
    for ticker in tickers:

        try:
            d = data[ticker]
            sector = d["sector"]
            avg = sector_averages[sector]
    
            pe_series = d.get("pe", [])
            ev_series = d.get("ev_ebitda", [])
            roic_series = d.get("roic", [])
            growth_series = d.get("revenue_growth", [])
            debt_ebitda_series = d.get("debt_ebitda", [])
            om_series = d.get("om", [])

            
            pe = pe_series[-1] if pe_series else None
            ev = ev_series[-1] if ev_series else None
            roic = roic_series[-1] if roic_series else None
            growth = growth_series[-1] if growth_series else None
            debt_ebitda = debt_ebitda_series[-1] if debt_ebitda_series else None
            om = om_series[-1] if om_series else None
            
            score = 0
            reasons = []
            
            # print('debt_ebitda',ticker,debt_ebitda,  avg["debt_ebitda"])

            # --- VALUATION ---
            if pe and avg["pe"] and pe < avg["pe"]:
                score += 1
                reasons.append("Low P/E vs sector avg")
                if trend_improving(pe_series, False):
                    score += 1
                    reasons.append("P/E decreasing")
                    
            if ticker == ticker_to_watch: # 1
                print(score)
                
            if ev and avg["ev_ebitda"] and ev < avg["ev_ebitda"]:
                score += 1
                reasons.append("Low EV/EBITDA vs sector avg")
                if trend_improving(ev_series, False):
                    score += 1
                    reasons.append("EV/EBITDA decreasing")

            if ticker == ticker_to_watch: # 2
                print(score)
                
            # --- QUALITY ---
            if roic and avg["roic"] and roic > avg["roic"]:
                score += 1
                reasons.append("High ROIC vs sector avg")
                if trend_improving(roic_series, True):
                    score += 1
                    reasons.append("ROIC increasing")
                    
            if ticker == ticker_to_watch: # 4
                print(score)
                
            # --- GROWTH ---
            if growth and growth > 0:
                score += 1
                reasons.append("Positive growth")
                if trend_improving(growth_series, True):
                    score += 1
                    reasons.append("Growth increasing")

            # if ticker == ticker_to_watch: #6
            #     print(ticker, score)

            # --- DEBT to EBITDA
            if debt_ebitda and debt_ebitda < avg['debt_ebitda']:
                score += 1
                reasons.append("Low Debt-to-EBITDA vs sector avg")
                if trend_improving(debt_ebitda_series, False):
                    score += 1
                    reasons.append("Debt-to-EBITDA decreasing")


            # --- Operating Margin
            if om and om > avg['om']:
                score += 1
                reasons.append("Higher OM vs sector avg")
                if trend_improving(om_series, True):
                    score += 1
                    reasons.append("OM increasing")
                    
            
            # --- RATING ---
            if score >= 8:
                rating = "STRONG BUY"
            elif score >= 6:
                rating = "BUY"
            elif score >= 3:
                rating = "HOLD"
            else:
                rating = "AVOID"

            
            results[ticker] = {
                "sector": sector,
                "score": score,
                "rating": rating,
                "reasons": reasons
            }

            # if ticker == ticker_to_watch:
            #     print (ticker, results[ticker])
                
            score = 0
            rating = ''    
            
        except Exception as e:
            print (f"Error in evaluate_stocks_by_sector for ticker: {ticker} {e}")

    return results

In [145]:

# df_3, dict_3 = get_formatted_data(industrial)
results = evaluate_stocks_by_sector(df_3.index.tolist(), dict_3)


In [146]:


values = [v for k, v in results.items()]
keys = [k for k, v in results.items()]
df_5 = pd.DataFrame({'key': keys, 'value': values})
buys = []
ratings = []
rois = []
scores = []
reasons = []
for index, row in df_5.iterrows():
    buys.append(row['key'])        
    ratings.append(row['value']['rating'])
    scores.append(row['value']['score'])
    reasons.append(row['value']['reasons'])

df_5 = pd.DataFrame({'ticker': buys, 'rating': ratings, 'score': scores, 'reason': reasons})
df_5.reset_index(inplace=True)
df_5.set_index(['ticker'], inplace=True)
df_5['roi'] = df_3['roi']
betas = pd.DataFrame(get_beta_5_year_for_df(df_5))
betas.set_index(['ticker'], inplace=True)
df_5['beta'] = betas['beta']
df_5["risk_bucket"] = pd.cut(
    df_5["beta"],
    bins=[-float("inf"), 0.8, 1.2, float("inf")],
    labels=["Low Risk", "Market-like", "High Risk"]
)
df_5['om'] = df_3['om'][:-1]



In [147]:
df_5[df_5['risk_bucket'] != 'High Risk'].sort_values(by=['score','rating','roi'],ascending=[False, False, False]).head(50)
# df_5.sort_values(by=['score','rating','roi'],ascending=[False, False, False]).to_csv('industrials.csv')

,index,rating,score,reason,roi,beta,risk_bucket,om
ticker,,,,,,,,
PAC,197,STRONG BUY,9,"[Low P/E vs sector avg, P/E decreasing, Low EV...",0.497408,0.98,Market-like,"[45.5176121317077, 44.4618364750747, 42.269935..."
SNA,230,STRONG BUY,9,"[Low P/E vs sector avg, P/E decreasing, Low EV...",0.056353,0.74,Low Risk,"[25.5642777440636, 26.2489233419466, 25.773355..."
ESEA,91,STRONG BUY,8,"[Low P/E vs sector avg, P/E decreasing, Low EV...",0.900154,0.57,Low Risk,"[57.7711818081052, 53.6991751511678, 57.027101..."
MLI,174,STRONG BUY,8,"[Low P/E vs sector avg, P/E decreasing, Low EV...",0.456121,1.14,Market-like,"[21.5378857980701, 20.3294393974049, 21.314657..."
ASR,20,STRONG BUY,8,"[Low P/E vs sector avg, Low EV/EBITDA vs secto...",0.398628,0.44,Low Risk,"[59.0339058194747, 55.9152494158915, 45.636566..."
NMM,185,STRONG BUY,8,"[Low P/E vs sector avg, P/E decreasing, Low EV...",0.132578,1.06,Market-like,"[34.2166779275057, 33.5568854914225, 31.990643..."
AIT,9,STRONG BUY,8,"[Low P/E vs sector avg, P/E decreasing, Low EV...",0.083970,0.86,Market-like,"[10.8672872561012, 11.4015786914604, 12.006971..."
CPRT,56,STRONG BUY,8,"[Low P/E vs sector avg, Low EV/EBITDA vs secto...",-0.304865,1.02,Market-like,"[38.4174204642542, 37.1038157600636, 36.512359..."
PPIH,208,BUY,7,"[Low P/E vs sector avg, P/E decreasing, Low EV...",1.095238,0.58,Low Risk,"[8.81275386943478, 12.8175825840994, 13.961834..."


In [112]:
def classify_beta(beta):
    if beta is None:
        return "Unknown"
    elif beta < 0.8:
        return "Low Risk"
    elif beta <= 1.2:
        return "Market-like"
    else:
        return "High Risk"

def get_beta(ticker):
    data, beta = None, None
    with open(f'{PATH}dividends/{ticker}.json', 'r') as file:
        data = json.load(file)['data']        
    
    beta = next((item for item in data if item["id"] == 'beta_5_year'), None)['value']        
    return beta
    
    
def get_beta_5_year_for_df(df):
    df_6 = df.copy()
    betas = []
    for index, row in df_6.iterrows():
        beta = get_beta(index)
        betas.append({'ticker': index, 'beta': round(beta, 2)})
    return betas
